# EfficientAD on MVTec AD — TPU (Kaggle TPU v5e-8, single chip)

Trains **EfficientAD** (Anomalib's lightweight teacher–student anomaly detector) on **MVTec AD** using **one** TPU chip. Runs in any notebook editor — Jupyter Lab, Jupyter Notebook, Kaggle, Colab, VSCode — wherever a TPU is reachable via `torch_xla`.

- **Tested runtime:** Kaggle Notebook → *Settings → Accelerator → TPU VM v5e-8*
- **Default category:** `bottle` (change `CATEGORY` in the config cell to swap)
- **Expected wall time:** ~10–15 min for one category on a single v5e chip
- A loop at the bottom runs all 15 categories and aggregates results into a DataFrame.

> **Why only one of the 8 chips?**
> Kaggle exposes TPU v5e-8 as an 8-worker pod slice. Initializing all 8 workers requires
> launching 8 host processes — that doesn't work from inside a single Jupyter kernel,
> and Lightning's XLA launcher fails with `Expected 8 worker addresses, got 1`.
> Setting `devices=1` makes Lightning run inline on chip 0 (no multi-process spawn),
> which works reliably. EfficientAD's `BATCH_SIZE=1` paper protocol benefits little
> from 8-way replication anyway.

In [ ]:
# Verify TPU is available — Kaggle TPU v5e-8 ships torch_xla pre-installed
import torch
import torch_xla
import torch_xla.core.xla_model as xm

print(f"torch     = {torch.__version__}")
print(f"torch_xla = {torch_xla.__version__}")
print(f"TPU device = {xm.xla_device()}")

In [ ]:
# Install anomalib.
# Kaggle TPU runtime ships torch + torch_xla pre-installed — do NOT upgrade them.
# Two known issues on this runtime:
#   1. The legacy `pytorch-lightning` package (if present) defines a different
#      LightningModule than anomalib's `lightning.pytorch.LightningModule`,
#      breaking the Trainer's isinstance check.
#   2. We must install anomalib with `--no-deps`-style restraint via pip flags
#      so it doesn't try to swap out the TPU-compatible torch build.
!pip uninstall -y -q pytorch-lightning
!pip install -q --upgrade "anomalib==2.4.1"

# After install: Runtime > Restart session, then re-run from this cell.

In [ ]:
# Imports and setup
import os, warnings, random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import pandas as pd
from PIL import Image

from anomalib.models import EfficientAd
from anomalib.data import MVTecAD as MVTec   # anomalib 2.x renamed MVTec → MVTecAD
from anomalib.engine import Engine

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Editor-agnostic output dir — relative to the kernel's working directory.
# Resolves to: /kaggle/working/results (Kaggle), /content/results (Colab),
# or ./results next to the notebook (Jupyter / VSCode / local).
RESULTS_DIR = Path("results").resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Results dir: {RESULTS_DIR}")

## MVTec AD Dataset

The MVTec AD dataset contains **15 categories** of industrial objects:

1. **bottle** - Glass bottles with various defects
2. **cable** - Power cables and connectors
3. **capsule** - Pharmaceutical capsules
4. **carpet** - Woven carpet samples
5. **grid** - Metal grids and mesh
6. **hazelnut** - Hazelnut kernels
7. **leather** - Leather surfaces
8. **metal_nut** - Metal fasteners
9. **pill** - Pharmaceutical pills
10. **screw** - Metal screws
11. **tile** - Floor tiles
12. **toothbrush** - Toothbrush bristles
13. **transistor** - Electronic transistors
14. **wood** - Wood plank samples
15. **zipper** - Zipper mechanisms

### Dataset Details
- **Total size**: ~5 GB
- **Image resolution**: 700×700 or 1024×1024 pixels (depending on category)
- **Annotations**: Pixel-level ground truth masks for defects
- **Auto-download**: Anomalib automatically downloads the dataset on first use
- **Download time**: ~2-3 minutes on Colab (depending on internet speed)

The dataset will be cached, so subsequent runs only download new categories.

In [ ]:
# Configuration — minimal.
# NUM_CORES = 1 by design: Kaggle's TPU v5e-8 is exposed as an 8-worker pod
# slice that can't be initialized from a single notebook kernel. We use one
# chip via Lightning's inline (non-spawn) path.
CATEGORY     = "bottle"   # 'bottle' / 'cable' / 'capsule' / 'carpet' / 'grid' / 'hazelnut' /
                          # 'leather' / 'metal_nut' / 'pill' / 'screw' / 'tile' / 'toothbrush' /
                          # 'transistor' / 'wood' / 'zipper'
MODEL_SIZE   = "small"    # "small" or "medium"
IMAGE_SIZE   = 256
BATCH_SIZE   = 1          # paper protocol for EfficientAD
NUM_EPOCHS   = 250
NUM_CORES    = 1          # one TPU v5e chip (do NOT set to 8 in a notebook)

# Editor-agnostic dataset dir — relative to the kernel's working directory
DATASET_ROOT = Path("mvtec").resolve()
DATASET_ROOT.mkdir(parents=True, exist_ok=True)

print(f"category={CATEGORY}  model={MODEL_SIZE}  cores={NUM_CORES}  epochs={NUM_EPOCHS}")

In [5]:
# Initialize datamodule
# anomalib 2.x splits batch_size into train_batch_size / eval_batch_size
print(f"Initializing MVTec datamodule for category: {CATEGORY}")
print("This may take a few minutes to download the dataset on first run...\n")

datamodule = MVTec(
    root=str(DATASET_ROOT),
    category=CATEGORY,
    train_batch_size=BATCH_SIZE,
    eval_batch_size=BATCH_SIZE,
    num_workers=2,
)

# Trigger download and prepare data
datamodule.prepare_data()
datamodule.setup()

# Print dataset statistics
train_loader = datamodule.train_dataloader()
test_loader  = datamodule.test_dataloader()

num_train_samples = len(train_loader.dataset)
num_test_samples  = len(test_loader.dataset)

print("Dataset Statistics")
print("-" * 50)
print(f"Category:        {CATEGORY}")
print(f"Training samples: {num_train_samples}")
print(f"Test samples:     {num_test_samples}")
print(f"Image size:       {IMAGE_SIZE}×{IMAGE_SIZE}")
print(f"Total samples:    {num_train_samples + num_test_samples}")

Initializing MVTec datamodule for category: bottle
This may take a few minutes to download the dataset on first run...



mvtecad: 5.26GB [04:32, 19.3MB/s]                               


Dataset Statistics
--------------------------------------------------
Category:        bottle
Training samples: 209
Test samples:     83
Image size:       256×256
Total samples:    292


In [6]:
# Initialize EfficientAD model
# anomalib 2.x: image_size is set on the datamodule, not the model constructor.
print(f"Initializing EfficientAD model (size: {MODEL_SIZE})...\n")

model = EfficientAd(model_size=MODEL_SIZE)

# Print model info
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model Information")
print("-" * 50)
print(f"Model:               EfficientAD ({MODEL_SIZE})")
print(f"Total parameters:    {total_params:,}")
print(f"Trainable parameters:{trainable_params:,}")

Initializing EfficientAD model (size: small)...

Model Information
--------------------------------------------------
Model:               EfficientAD (small)
Total parameters:    8,058,628
Trainable parameters:8,058,628


In [ ]:
# Training on a single TPU v5e chip
# (NUM_CORES is fixed to 1 — see the warning in the config cell)
import sys
sys.setrecursionlimit(10000)

print(f"Training '{CATEGORY}' on TPU v5e ({NUM_CORES} chip) for {NUM_EPOCHS} epochs")

engine = Engine(
    accelerator="tpu",
    devices=NUM_CORES,
    max_epochs=NUM_EPOCHS,
    default_root_dir=str(RESULTS_DIR),
    num_sanity_val_steps=0,
    limit_val_batches=0,
    check_val_every_n_epoch=None,
)
engine.fit(model=model, datamodule=datamodule)
print("✓ Training complete")

In [ ]:
# Evaluation — reuse the engine from the training cell
print(f"Evaluating model on '{CATEGORY}' test set...\n")

try:
    metrics = engine.test(model=model, datamodule=datamodule)

    print("Evaluation Results")
    print("=" * 60)

    if metrics:
        metric_dict = metrics[0] if isinstance(metrics, list) else metrics

        # Highlight the headline metrics first
        for headline_key, label in [
            ("image_AUROC",  "Image-level AUROC"),
            ("image_F1Max",  "Image F1 Max"),
            ("pixel_AUROC",  "Pixel-level AUROC (localization)"),
            ("pixel_F1Max",  "Pixel F1 Max"),
        ]:
            if headline_key in metric_dict:
                print(f"  {label:<35} {metric_dict[headline_key]:.4f}")

        print("\nAll metrics returned by the engine:")
        print("-" * 60)
        for k, v in metric_dict.items():
            if isinstance(v, (int, float)):
                print(f"  {k}: {v:.4f}")
    else:
        print("No metrics returned. Check the training logs.")

except Exception as e:
    print(f"Error during evaluation: {e}")
    metrics = None

## Visualization of Predictions

Below we visualize model predictions on test images, showing:
- **Original image**: The raw test image
- **Ground truth mask**: Pixel-level defect annotation (if available)
- **Anomaly heatmap**: Model's confidence that each pixel is anomalous
- **Binary prediction**: Thresholded anomaly map (0 = normal, 1 = anomaly)

We display a mix of normal and defective samples to show how the model distinguishes between them.

In [ ]:
# Visualization of predictions
# anomalib 2.x: engine.predict() returns a list of Batch objects, each Batch can
# contain multiple samples (B>1). With BATCH_SIZE=1 each batch is one sample.
print("Generating predictions for visualization...")

try:
    predictions = engine.predict(model=model, datamodule=datamodule)

    # Flatten batched predictions → per-sample dicts so the indexing below is simple
    samples = []
    for batch in predictions:
        bsz = batch.image.shape[0]
        for i in range(bsz):
            samples.append({
                "image":       batch.image[i].detach().cpu().numpy(),
                "gt_mask":     batch.gt_mask[i].detach().cpu().numpy()
                                  if getattr(batch, "gt_mask", None) is not None else None,
                "gt_label":    int(batch.gt_label[i].item())
                                  if getattr(batch, "gt_label", None) is not None else 0,
                "anomaly_map": batch.anomaly_map[i].detach().cpu().numpy(),
                "pred_score":  float(batch.pred_score[i].item())
                                  if getattr(batch, "pred_score", None) is not None else 0.0,
            })

    if not samples:
        raise RuntimeError("engine.predict returned no samples")

    # Pick a spread of normal + defective samples
    num_display = min(6, len(samples))
    indices = np.linspace(0, len(samples) - 1, num_display, dtype=int)

    fig, axes = plt.subplots(num_display, 4, figsize=(16, 4 * num_display))
    if num_display == 1:
        axes = axes.reshape(1, -1)

    for row, idx in enumerate(indices):
        s = samples[idx]
        gt_label_str = "DEFECTIVE" if s["gt_label"] == 1 else "NORMAL"

        # --- Original image (CHW → HWC, clipped to [0,1]) ---
        img = s["image"]
        if img.ndim == 3 and img.shape[0] in (1, 3):
            img = np.transpose(img, (1, 2, 0))
        img = np.clip(img, 0, 1)
        axes[row, 0].imshow(img.squeeze(), cmap="gray" if img.ndim == 2 else None)
        axes[row, 0].set_title(f"Image (GT: {gt_label_str})", fontweight="bold")
        axes[row, 0].axis("off")

        # --- Ground truth mask ---
        gt_mask = s["gt_mask"]
        if gt_mask is not None and gt_mask.size > 0:
            axes[row, 1].imshow(gt_mask.squeeze(), cmap="binary")
            axes[row, 1].set_title("Ground Truth Mask", fontweight="bold")
        else:
            axes[row, 1].text(0.5, 0.5, "no mask", ha="center", va="center",
                              transform=axes[row, 1].transAxes)
            axes[row, 1].set_title("Ground Truth Mask", fontweight="bold")
        axes[row, 1].axis("off")

        # --- Anomaly heatmap ---
        amap = s["anomaly_map"].squeeze()
        im = axes[row, 2].imshow(amap, cmap="inferno")
        axes[row, 2].set_title(f"Anomaly Heatmap (score={s['pred_score']:.3f})",
                                fontweight="bold")
        axes[row, 2].axis("off")
        plt.colorbar(im, ax=axes[row, 2], fraction=0.046, pad=0.04)

        # --- Binary prediction ---
        threshold = float(amap.mean() + amap.std())   # sample-relative threshold
        binary = (amap > threshold).astype(float)
        axes[row, 3].imshow(binary, cmap="binary")
        axes[row, 3].set_title(f"Binary (τ={threshold:.3f})", fontweight="bold")
        axes[row, 3].axis("off")

    plt.tight_layout()
    out_path = RESULTS_DIR / f"predictions_{CATEGORY}.png"
    plt.savefig(out_path, dpi=100, bbox_inches="tight")
    plt.show()
    print(f"\n✓ Visualization saved to {out_path}")

except Exception as e:
    print(f"Error during visualization: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Inference on a single image (TPU)
import torch_xla.core.xla_model as xm
device = xm.xla_device()

test_loader  = datamodule.test_dataloader()
test_batch   = next(iter(test_loader))
image_tensor = test_batch.image[:1].to(device)
gt_label     = int(test_batch.gt_label[0].item()) if getattr(test_batch, "gt_label", None) is not None else None

model.eval()
model = model.to(device)
with torch.no_grad():
    output = model(image_tensor)
xm.mark_step()   # flush XLA graph

anomaly_score = float(output.pred_score[0].item())
anomaly_map   = output.anomaly_map[0].detach().cpu().numpy().squeeze()
threshold     = float(anomaly_map.mean() + anomaly_map.std())
binary        = (anomaly_map > threshold).astype(int)

print(f"GT: {'DEFECTIVE' if gt_label == 1 else 'NORMAL' if gt_label == 0 else 'unknown'}")
print(f"Anomaly score : {anomaly_score:.4f}")
print(f"Map shape     : {anomaly_map.shape}")
print(f"Threshold τ   : {threshold:.4f}")

img = image_tensor[0].detach().cpu().numpy()
if img.shape[0] in (1, 3):
    img = np.transpose(img, (1, 2, 0))
img = np.clip(img, 0, 1).squeeze()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img, cmap="gray" if img.ndim == 2 else None)
axes[0].set_title("Input"); axes[0].axis("off")
im = axes[1].imshow(anomaly_map, cmap="inferno")
axes[1].set_title(f"Anomaly Map (score={anomaly_score:.3f})"); axes[1].axis("off")
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
axes[2].imshow(binary, cmap="binary")
axes[2].set_title(f"Binary (τ={threshold:.3f})"); axes[2].axis("off")
plt.tight_layout()
plt.savefig(RESULTS_DIR / f"single_inference_{CATEGORY}.png", dpi=100, bbox_inches="tight")
plt.show()

## Running on All Categories

You can loop through all 15 MVTec AD categories and collect results in a summary table. This will take approximately 3-4 hours on a T4 GPU.

The code below is commented out by default. To run it:
1. Set `RUN_ALL_CATEGORIES = True` in the next cell
2. Or uncomment the loop and run it separately
3. Results will be collected in a pandas DataFrame with per-category metrics

In [ ]:
# Run all 15 categories on TPU v5e-8
RUN_ALL_CATEGORIES = False   # ← flip to True

ALL_CATEGORIES = [
    'bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut',
    'leather', 'metal_nut', 'pill', 'screw', 'tile', 'toothbrush',
    'transistor', 'wood', 'zipper',
]

if RUN_ALL_CATEGORIES:
    results = []
    for i, category in enumerate(ALL_CATEGORIES):
        print(f"\n[{i+1}/{len(ALL_CATEGORIES)}] {category}")
        try:
            dm = MVTec(
                root=str(DATASET_ROOT), category=category,
                train_batch_size=BATCH_SIZE, eval_batch_size=BATCH_SIZE, num_workers=2,
            )
            dm.prepare_data(); dm.setup()
            mdl = EfficientAd(model_size=MODEL_SIZE)
            eng = Engine(
                accelerator="tpu", devices=NUM_CORES,
                max_epochs=NUM_EPOCHS,
                default_root_dir=str(RESULTS_DIR / category),
                num_sanity_val_steps=0, enable_progress_bar=False,
                limit_val_batches=0, check_val_every_n_epoch=None,
            )
            eng.fit(model=mdl, datamodule=dm)
            test_metrics = eng.test(model=mdl, datamodule=dm)
            md = test_metrics[0] if isinstance(test_metrics, list) else test_metrics
            row = {"Category": category}
            for k in ["image_AUROC", "image_F1Max", "pixel_AUROC", "pixel_F1Max"]:
                row[k] = (md or {}).get(k, float("nan"))
            results.append(row)
        except Exception as e:
            print(f"✗ {category}: {e}")
            results.append({"Category": category, "Status": "Failed"})

    df = pd.DataFrame(results)
    print("\n" + df.to_string(index=False))
    df.to_csv(RESULTS_DIR / "all_categories_results.csv", index=False)
    print(f"\n✓ Saved → {RESULTS_DIR / 'all_categories_results.csv'}")
else:
    print("Set RUN_ALL_CATEGORIES = True to run all 15 categories.")

## Using Your Own Data

To run EfficientAD on your custom dataset, organize your images in this folder structure:

```
your_dataset/
├── train/
│   └── good/          ← normal training images (no defects needed for training)
│       ├── image_001.jpg
│       └── ...
└── test/
    ├── good/          ← normal test images
    │   └── ...
    └── defective/     ← defective test images
        ├── crack_001.jpg
        ├── crack_001_mask.png   ← optional pixel-level ground truth (grayscale)
        └── ...
```

### Key Points
- **Training** only needs **normal (good)** images — EfficientAD is unsupervised
- **Ground truth masks** are optional; without them you still get image-level AUROC
- **Image format**: PNG or JPG, any resolution (resized to `image_size` internally)
- **Minimum samples**: 50–100 normal images recommended

### Custom Dataset Example (anomalib 2.x API)

```python
from anomalib.data import Folder
from anomalib.models import EfficientAd
from anomalib.engine import Engine

custom_dm = Folder(
    root="/path/to/your_dataset",
    normal_dir="train/good",
    abnormal_dir="test/defective",
    normal_test_dir="test/good",
    mask_dir="test/defective",      # omit if no masks
    image_size=256,
    train_batch_size=1,
    eval_batch_size=1,
)
custom_dm.prepare_data()
custom_dm.setup()

model = EfficientAd(model_size="small")   # no image_size param in 2.x

engine = Engine(
    task="segmentation",                  # use "classification" if no masks
    accelerator="gpu",
    devices=1,
    max_epochs=250,
)
engine.fit(model=model, datamodule=custom_dm)
engine.test(model=model, datamodule=custom_dm)
```

### Hyperparameter Guide

| Parameter | Recommended | Notes |
|-----------|-------------|-------|
| `max_epochs` | 250 | Anomalib default; tuned for MVTec-sized datasets |
| `train_batch_size` | 1 | Safe for T4 16 GB |
| `image_size` | 256 | Balance between quality and speed |
| `model_size` | `"small"` | Start here; `"medium"` is more accurate but slower |

### Tips
1. Consistent lighting and camera angle dramatically affect accuracy
2. More normal training images → better background model
3. Save the trained model checkpoint from `RESULTS_DIR` for deployment
4. Tune the anomaly threshold on a held-out validation set to match your precision/recall target